# 01 - Preprocesamiento y Definición del Problema y Dataset

**Entregable 2 - Deep Learning**  
Estado del Arte, EDA Mejorado e Implementación de Modelos Benchmark

## 1. Definición del Problema y Dataset

### Definición precisa del problema
El objetivo es desarrollar modelos de **segmentación semántica** (pixel-level classification) capaces de identificar automáticamente el tipo de terreno en imágenes capturadas por rovers de la NASA en la superficie de Marte. 

Las clases a segmentar son:
- **0 - Suelo (Soil)**
- **1 - Roca (Bedrock)**
- **2 - Arena (Sand)**
- **3 - Rocas sueltas (Big Rocks)**
- **255 - Ignorar** (máscara de ignore / no etiquetada)

Esta tarea es crítica para la **navegación autónoma** de rovers, ya que un error en la clasificación (por ejemplo, confundir arena profunda con suelo firme) puede dejar atrapado permanentemente al rover.

### Justificación del dataset
Utilizamos el dataset **AI4MARS (Artificial Intelligence for Mars Rover Surveys)**, disponible en Zenodo (ID: 15995036) y curado por el **Jet Propulsion Laboratory (JPL) de la NASA**.

- **Misiones incluidas**: Mars Science Laboratory (Curiosity), Mars Exploration Rovers (Spirit y Opportunity) y Mars 2020 (Perseverance).
- **Instrumentos**: Principalmente cámaras de ingeniería (HazCams y NavCams).
- **Etiquetado**: Más de 326.000 etiquetas generadas por crowdsourcing + subconjunto de validación "gold" verificado por científicos de misión.

Este es el dataset más grande y representativo disponible para segmentación semántica en terreno marciano.

### Descripción técnica

- **Tamaño**: 23.928 muestras originales → **22.965 muestras válidas** después de limpieza.
- **Estructura**: Imágenes RAW + máscaras de segmentación semántica (un archivo PNG por imagen).
- **Tipo de datos**: 
  - Imágenes: Escala de grises y color (RGB), resolución variable (principalmente 1024×1024 y 1288×968).
  - Máscaras: PNG de un solo canal con valores {0, 1, 2, 3, 255}.
- **Variable objetivo**: Máscara de segmentación semántica (clasificación a nivel de píxel).

In [1]:
import os
import pandas as pd
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==================== CONFIGURACIÓN ====================
BASE = Path().resolve()
PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Directorios configurados")

✅ Directorios configurados


In [2]:
def canonical_name(name):
    name = name.replace("_0LLJ", "")
    for i in range(1, 100):
        name = name.replace(f"_merged{i}", "")
    return name

def build_manifest(image_dir, label_dir, mission, camera):
    image_dir = Path(image_dir)
    label_dir = Path(label_dir)

    images = {}
    for img in image_dir.iterdir():
        if img.suffix.lower() in [".jpg", ".jpeg"]:
            key = canonical_name(img.stem)
            if key not in images or img.suffix.lower() == ".jpeg":
                images[key] = img

    labels = {}
    for lbl in label_dir.iterdir():
        if lbl.suffix.lower() == ".png":
            key = canonical_name(lbl.stem)
            labels[key] = lbl

    keys = set(images.keys()) & set(labels.keys())

    rows = []
    for k in keys:
        rows.append({
            "mission": mission,
            "camera": camera,
            "id": k,
            "image_path": str(images[k]),
            "mask_path": str(labels[k])
        })

    return pd.DataFrame(rows)


# Construcción del manifest completo
msl_df = build_manifest(
    BASE / "msl" / "ncam" / "images" / "edr",
    BASE / "msl" / "ncam" / "labels" / "train",
    mission="MSL", camera="ncam"
)

mer_df = build_manifest(
    BASE / "mer" / "images" / "eff",
    BASE / "mer" / "labels" / "train" / "merged-unmasked",
    mission="MER", camera="ncam"
)

m2020_df = build_manifest(
    BASE / "m2020" / "images" / "ncam",
    BASE / "m2020" / "labels" / "NAV",
    mission="M2020", camera="ncam"
)

df = pd.concat([msl_df, mer_df, m2020_df], ignore_index=True)

print(f"Total muestras originales: {len(df)}")
print(df["mission"].value_counts())

Total muestras originales: 23928
mission
MSL      16064
MER       7391
M2020      473
Name: count, dtype: int64


Se limpian mascaras invalidas

In [3]:
def check_masks(df):
    empty_masks = []
    only_ignore = []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Verificando máscaras"):
        mask = np.array(Image.open(row["mask_path"]))
        unique = np.unique(mask)
        
        if len(unique) == 1:
            empty_masks.append(idx)
        if set(unique) == {255}:
            only_ignore.append(idx)
    
    return empty_masks, only_ignore

empty_masks, only_ignore = check_masks(df)

bad_indices = set(empty_masks) | set(only_ignore)
df_clean = df.drop(index=bad_indices).reset_index(drop=True)

print(f"\nMáscaras eliminadas: {len(bad_indices)}")
print(f"Dataset final limpio: {len(df_clean)} muestras")

Verificando máscaras: 100%|██████████| 23928/23928 [04:04<00:00, 97.75it/s] 


Máscaras eliminadas: 963
Dataset final limpio: 22965 muestras


In [4]:
df_clean.to_csv(PROCESSED_DIR / "manifest_clean.csv", index=False)
print(f"✅ Manifest guardado en: {PROCESSED_DIR / 'manifest_clean.csv'}")

# Estadísticas finales
print("\n=== Estadísticas finales del dataset ===")
print(f"Total imágenes válidas: {len(df_clean):,}")
print(df_clean["mission"].value_counts())

✅ Manifest guardado en: processed\manifest_clean.csv

=== Estadísticas finales del dataset ===
Total imágenes válidas: 22,965
mission
MSL      15901
MER       6593
M2020      471
Name: count, dtype: int64


## Decisión de preprocesamiento derivada

- Se eliminaron 963 máscaras vacías o que contenían únicamente píxeles de ignore (255).
- Se unificaron nombres de archivos para evitar duplicados.
- Se creó un `manifest_clean.csv` con rutas absolutas y metadata por misión.
- **Próximos pasos**: Resize a 256×256, normalización y data augmentation (se definirá en el EDA y en los notebooks de modelos).

**Impacto en la arquitectura del modelo**:  
Se trabajará con imágenes de tamaño fijo (256×256) → compatible con U-Net, DeepLabV3, SegFormer, etc.

**Riesgos identificados**:  
Riesgo de data leakage si no se hace split estratificado por misión → se controlará en el EDA.